# 03 · Create or update the daily Databricks Right-check job

Run once locally after editing Notebook 02 or changing the schedule. Notebook 01 writes local immutable events and the Windows wrapper syncs them to the External Volume backing prefix. This job first ingests those files, then completes their Right checks.


In [ ]:
PROFILE = "corp"                                  # ~/.databrickscfg
NOTEBOOK = "02_dbx_count_watch.ipynb"
WORKSPACE_DIR = None                              # None = /Users/<me>/count_watch
JOB_NAME = "count-watch-right"
CRON = "0 30 10 * * ?"                            # daily at 10:30
TIMEZONE = "America/New_York"
EMAIL = ""                                        # optional failure notification
WIDGETS = {
    "TABLE_LIST": "<catalog>.<schema>.table_list",
    "RECORDS_PATH": "/Volumes/<catalog>/<schema>/<volume>/count-watch",
    "LOOKBACK_DAYS": "7",
}
RUN_NOW = False


In [ ]:
from datetime import timedelta
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs
from databricks.sdk.service.workspace import ImportFormat

client = WorkspaceClient(profile=PROFILE)
me = client.current_user.me().user_name
folder = WORKSPACE_DIR or f"/Users/{me}/count_watch"
notebook_path = f"{folder}/{Path(NOTEBOOK).stem}"

client.workspace.mkdirs(folder)
client.workspace.upload(
    notebook_path,
    Path(NOTEBOOK).read_bytes(),
    format=ImportFormat.JUPYTER,
    overwrite=True,
)
print("uploaded", notebook_path)


In [ ]:
task = jobs.Task(
    task_key="right_check",
    notebook_task=jobs.NotebookTask(
        notebook_path=notebook_path,
        base_parameters=WIDGETS,
        source=jobs.Source.WORKSPACE,
    ),
)

settings = jobs.JobSettings(
    name=JOB_NAME,
    tasks=[task],
    schedule=jobs.CronSchedule(
        quartz_cron_expression=CRON,
        timezone_id=TIMEZONE,
        pause_status=jobs.PauseStatus.UNPAUSED,
    ),
    email_notifications=jobs.JobEmailNotifications(on_failure=[EMAIL]) if EMAIL else None,
    max_concurrent_runs=1,
)

existing = list(client.jobs.list(name=JOB_NAME))
if existing:
    job_id = existing[0].job_id
    client.jobs.reset(job_id=job_id, new_settings=settings)
else:
    job_id = client.jobs.create(
        name=settings.name,
        tasks=settings.tasks,
        schedule=settings.schedule,
        email_notifications=settings.email_notifications,
        max_concurrent_runs=settings.max_concurrent_runs,
    ).job_id

print("job", job_id, f"{client.config.host}/jobs/{job_id}")


In [ ]:
if RUN_NOW:
    run = client.jobs.run_now(job_id=job_id).result(timeout=timedelta(minutes=30))
    task_run = next(task for task in run.tasks if task.task_key == "right_check")
    print(task_run.state.result_state, task_run.state.state_message or "")
    output = client.jobs.get_run_output(task_run.run_id)
    print(output.notebook_output.result if output.notebook_output else "no notebook result")
